In [1]:
#!unzip /content/dual_training.zip -d /content/dual_training/

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0022.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0023.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0024.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0025.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0026.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0027.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0028.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0029.jpg  
  inflating: /content/dual_training/dual_training/dual_dataset/detection/test/images/glass_0030.jpg  
  inflating: /content/dual_training

In [2]:
!pip install ultralytics -q

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
import zipfile
import random
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter
import pandas as pd

print(f"🚀 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 119.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultra

In [8]:
# ==================== ÇIFTE EĞİTİM ====================
class DualTrainer:
    def __init__(self):
        self.classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
        self.detection_model = None
        self.classification_model = None

    def train_detection(self, yaml_path, epochs=50):
        """Detection modeli eğit"""
        print("🎯 DETECTION EĞİTİMİ BAŞLIYOR...")
        print("="*40)

        self.detection_model = YOLO('yolov8n.pt')

        results = self.detection_model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=640,
            batch=16,
            patience=20,
            device=0 if torch.cuda.is_available() else 'cpu',
            project='dual_training',
            name='detection',

            # Optimizasyon
            optimizer='AdamW',
            lr0=0.01,
            momentum=0.937,
            weight_decay=0.0005,

            # Augmentation
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=5.0,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,

            plots=True,
            save=True
        )

        print("✅ Detection eğitimi tamamlandı!")
        return results

    def train_classification(self, yaml_path, epochs=50):
        """Classification modeli eğit"""
        print("\n🏷️ CLASSIFICATION EĞİTİMİ BAŞLIYOR...")
        print("="*45)

        self.classification_model = YOLO('yolov8n-cls.pt')

        results = self.classification_model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=224,
            batch=32,
            patience=20,
            device=0 if torch.cuda.is_available() else 'cpu',
            project='dual_training',
            name='classification',

            # Optimizasyon
            optimizer='AdamW',
            lr0=0.01,
            momentum=0.937,
            weight_decay=0.0005,

            plots=True,
            save=True
        )

        print("✅ Classification eğitimi tamamlandı!")
        return results

    def load_models(self, det_path=None, cls_path=None):
        """Eğitilmiş modelleri yükle"""
        det_path = det_path or "dual_training/detection/weights/best.pt"
        cls_path = cls_path or "dual_training/classification/weights/best.pt"

        print(f"📥 Modeller yükleniyor...")

        self.detection_model = YOLO(det_path)
        self.classification_model = YOLO(cls_path)

        print("✅ Modeller yüklendi!")

    def predict_ensemble(self, image_path, det_conf=0.5, alpha=0.7):
        """
        Ensemble tahmin yap

        Args:
            alpha: Detection ağırlığı (0-1)
        """
        if not self.detection_model or not self.classification_model:
            print("❌ Modeller yüklenmemiş!")
            return None

        # Detection
        det_results = self.detection_model(image_path, conf=det_conf, verbose=False)

        # Classification
        cls_results = self.classification_model(image_path, verbose=False)

        # Detection sonuçları
        det_conf_score = 0
        det_class_idx = -1

        if len(det_results[0].boxes) > 0:
            best_idx = torch.argmax(det_results[0].boxes.conf)
            det_conf_score = det_results[0].boxes.conf[best_idx].item()
            det_class_idx = int(det_results[0].boxes.cls[best_idx].item())

        # Classification sonuçları
        cls_class_idx = cls_results[0].probs.top1
        cls_conf_score = cls_results[0].probs.top1conf.item()

        # Ensemble karar
        if det_class_idx == cls_class_idx and det_class_idx >= 0:
            # Aynı sınıf - güven skorlarını birleştir
            final_conf = alpha * det_conf_score + (1 - alpha) * cls_conf_score
            final_class = det_class_idx
            agreement = True
        else:
            # Farklı sınıflar - daha güvenilir olanı seç
            if det_class_idx >= 0 and alpha * det_conf_score > (1 - alpha) * cls_conf_score:
                final_class = det_class_idx
                final_conf = det_conf_score
            else:
                final_class = cls_class_idx
                final_conf = cls_conf_score
            agreement = False

        return {
            'final_class': self.classes[final_class],
            'final_confidence': final_conf,
            'detection': {'class': self.classes[det_class_idx] if det_class_idx >= 0 else 'none',
                         'confidence': det_conf_score},
            'classification': {'class': self.classes[cls_class_idx],
                             'confidence': cls_conf_score},
            'agreement': agreement,
            'all_probs': cls_results[0].probs.data.cpu().numpy()
        }


In [12]:
# ==================== YAML DOSYALARINI DÜZELT ====================
import yaml
from pathlib import Path

def fix_yaml_paths():
    """YAML dosyalarındaki path'leri Colab için düzelt"""

    # Detection YAML düzelt
    det_yaml_path = '/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml'

    with open(det_yaml_path, 'r') as f:
        det_config = yaml.safe_load(f)

    # Path'i Colab'a uygun hale getir
    det_config['path'] = '/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml'
    det_config['train'] = 'train/images'
    det_config['val'] = 'val/images'
    det_config['test'] = 'test/images'

    with open(det_yaml_path, 'w') as f:
        yaml.dump(det_config, f)

    print("✅ Detection YAML düzeltildi")

    # Classification YAML düzelt
    cls_yaml_path = '/content/dual_training/dual_training/dual_dataset/classification/dataset.yaml'

    with open(cls_yaml_path, 'r') as f:
        cls_config = yaml.safe_load(f)

    # Path'i Colab'a uygun hale getir
    cls_config['path'] = '/content/dual_training/dual_training/dual_dataset/classification/dataset.yaml'
    cls_config['train'] = 'train'
    cls_config['val'] = 'val'
    cls_config['test'] = 'test'

    with open(cls_yaml_path, 'w') as f:
        yaml.dump(cls_config, f)

    print("✅ Classification YAML düzeltildi")

    # Klasör yapısını kontrol et
    det_base = Path('/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml')
    cls_base = Path('/content/dual_training/dual_training/dual_dataset/classification/dataset.yaml')

    print(f"\n📁 KLASÖR YAPISI KONTROLÜ:")
    print(f"Detection base: {det_base.exists()}")
    print(f"Detection train/images: {(det_base / 'train/images').exists()}")
    print(f"Detection val/images: {(det_base / 'val/images').exists()}")
    print(f"Detection test/images: {(det_base / 'test/images').exists()}")

    print(f"\nClassification base: {cls_base.exists()}")
    print(f"Classification train: {(cls_base / 'train').exists()}")
    print(f"Classification val: {(cls_base / 'val').exists()}")
    print(f"Classification test: {(cls_base / 'test').exists()}")

    # Görüntü sayılarını kontrol et
    if (det_base / 'train/images').exists():
        train_count = len(list((det_base / 'train/images').glob('*.jpg')))
        val_count = len(list((det_base / 'val/images').glob('*.jpg')))
        test_count = len(list((det_base / 'test/images').glob('*.jpg')))
        print(f"\n📊 Detection görüntü sayıları:")
        print(f"Train: {train_count}, Val: {val_count}, Test: {test_count}")

# YAML'leri düzelt
fix_yaml_paths()

✅ Detection YAML düzeltildi
✅ Classification YAML düzeltildi

📁 KLASÖR YAPISI KONTROLÜ:
Detection base: True
Detection train/images: False
Detection val/images: False
Detection test/images: False

Classification base: True
Classification train: False
Classification val: False
Classification test: False


In [13]:
# Detection modeli oluştur
detection_model = YOLO('/content/yolov8n.pt')

# Detection eğitimi
detection_results = detection_model.train(
    data='/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='trash_training',
    name='detection_model',
    save=True,
    plots=True,
 # Optimizasyon
    optimizer='AdamW',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005
)

print("✅ Detection eğitimi tamamlandı!")
print(f"📁 Model kaydedildi: trash_training/detection_model/weights/best.pt")

Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=detection_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pa

RuntimeError: Dataset '/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml' error ❌ Dataset '/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml' images not found, missing path '/content/dual_training/dual_training/dual_dataset/detection/dataset.yaml/val/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'